In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score
from tqdm import tqdm
import glob

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")  # 強制的にCPUに

print(device)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

def compute_mean_std(data_dir):
    all_values = []
    data_files = sorted([f for f in glob.glob(os.path.join(data_dir, "*.csv")) if "label" not in os.path.basename(f)])

    for f in data_files:
        df = pd.read_csv(f)
        dq_dv = df.iloc[:, 1].values.astype("float32")  # dQ/dV列
        dq_dv = np.nan_to_num(dq_dv, nan=0.0, posinf=1.0, neginf=-1.0)  # 安全処理
        all_values.append(dq_dv)

    all_concat = np.concatenate(all_values, axis=0)
    mean = all_concat.mean()
    std = all_concat.std()

    return mean, std

class CSVSequenceDataset(Dataset):
    def __init__(self, data_dir, mean=None, std=None):
        self.data_dir = data_dir
        self.csv_files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
        self.data_files = [f for f in self.csv_files if "label" not in os.path.basename(f)]
        self.label_files = [f.replace(".csv", "_label.csv") for f in self.data_files]

        # 正規化パラメータ（Noneの場合は未適用）
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.data_files)

    def __getitem__(self, idx):
        data_path = self.data_files[idx]
        label_path = self.label_files[idx]

        df = pd.read_csv(data_path)
        dq_dv = df.iloc[:, 1].values.astype("float32")  # dQ/dV列

        # NaNやinfを処理（正規化前）
        dq_dv = np.nan_to_num(dq_dv, nan=0.0, posinf=1.0, neginf=-1.0)

        if self.mean is not None and self.std is not None:
            if self.std == 0:
                print(f"[!] Warning: std=0 for normalization, fallback to std=1")
                self.std = 1.0
            dq_dv = (dq_dv - self.mean) / self.std

        # 入力の検査（オプション）
        if np.any(np.isnan(dq_dv)) or np.any(np.isinf(dq_dv)):
            print(f"[!] Warning: NaN or inf in input at {data_path}")

        dq_dv = torch.tensor(dq_dv).unsqueeze(-1)  # [seq_len, 1]

        label = pd.read_csv(label_path, skiprows=1, header=None).values.flatten().astype("float32")
        label = np.clip(label, 0, 1)
        label = np.nan_to_num(label, nan=0.0, posinf=1.0, neginf=0.0)  # ラベルにも対応
        label = torch.tensor(label, dtype=torch.float32)

        return dq_dv, label


In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, input_dim=1, model_dim=64, num_heads=4, num_layers=2, num_classes=7):
        super().__init__()
        self.embedding = nn.Linear(input_dim, model_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),  # [batch, model_dim, 1]
            nn.Flatten(),             # [batch, model_dim]
            nn.Linear(model_dim, num_classes),
            nn.Sigmoid()              # マルチラベル用
        )

    def forward(self, x):  # x: [batch, seq_len, input_dim]
        x = self.embedding(x)          # [batch, seq_len, model_dim]
        x = x.permute(1, 0, 2)         # [seq_len, batch, model_dim]
        x = self.transformer_encoder(x)
        x = x.permute(1, 2, 0)         # [batch, model_dim, seq_len]
        return self.classifier(x)

In [ ]:
# === データローダー ===
batch_size = 128
data_root = "D:/Yamato/dQdV"

# datasets = {
#     split: CSVSequenceDataset(os.path.join(data_root, split))
#     for split in ['Train', 'Val', 'Test']
# }


# Train データのみから mean, std を計算
mean, std = compute_mean_std(os.path.join(data_root, "Train"))

# 各データセットに同じ mean, std を渡す
datasets = {
    split: CSVSequenceDataset(os.path.join(data_root, split), mean=mean, std=std)
    for split in ['Train', 'Val', 'Test']
}




dataloaders = {
    split: DataLoader(datasets[split], batch_size=batch_size, shuffle=(split == 'Train'))
    for split in ['Train', 'Val', 'Test']
}

# === モデル定義 ===
model = TransformerClassifier(input_dim=1, model_dim=64, num_heads=4, num_layers=2, num_classes=7).to(device)

# === ロス関数・最適化手法 ===
# criterion = nn.BCEWithLogitsLoss()
criterion = nn.BCELoss()

optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# === 学習ループ ===
num_epochs = 5000
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    for phase in ['Train', 'Val']:
        model.train() if phase == 'Train' else model.eval()
        running_loss = 0.0
        all_preds = []
        all_labels = []

        for inputs, labels in tqdm(dataloaders[phase]):
            inputs = inputs.to(device)  # (B, T, 1)
            labels = labels.to(device)  # (B, 7)

            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == 'Train'):
                outputs = model(inputs)  # (B, 7)
#                 print("outputs:", outputs.detach().cpu().numpy())
#                 print("labels:", labels.detach().cpu().numpy())
                loss = criterion(outputs, labels)
                if phase == 'Train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            preds = (outputs > 0.5).int().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            
        save_path = "Transformer_dqdv_classifierBCELoss.pth"
        torch.save(model.state_dict(), save_path)
        epoch_loss = running_loss / len(datasets[phase])
        epoch_f1 = f1_score(all_labels, all_preds, average='micro')
        print(f"{phase} Loss: {epoch_loss:.4f} F1: {epoch_f1:.4f}")

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, hamming_loss
import numpy as np
from tqdm import tqdm

# === モデル読み込み ===
model.load_state_dict(torch.load("Transformer_dqdv_classifierBCELoss.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in tqdm(dataloaders['Test']):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = (outputs > 0.5).int().cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

# === 評価指標の計算 ===
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

f1_micro = f1_score(all_labels, all_preds, average='micro')
f1_macro = f1_score(all_labels, all_preds, average='macro')
precision_macro = precision_score(all_labels, all_preds, average='macro', zero_division=0)
recall_macro = recall_score(all_labels, all_preds, average='macro', zero_division=0)
subset_acc = accuracy_score(all_labels, all_preds)
hamming = hamming_loss(all_labels, all_preds)

# === 結果表示 ===
print(f"F1 Score (micro): {f1_micro:.4f}")
print(f"F1 Score (macro): {f1_macro:.4f}")
print(f"Precision (macro): {precision_macro:.4f}")
print(f"Recall (macro): {recall_macro:.4f}")
print(f"Subset Accuracy: {subset_acc:.4f}")
print(f"Hamming Loss: {hamming:.4f}")
